In [ ]:
from pathlib import Path
import tempfile
import zipfile

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.transform import array_bounds
from rasterio.windows import Window, from_bounds


zip_path = next(Path("Output/s1").glob("*.zip"))
figure_folder = Path("Output/figures")
figure_folder.mkdir(parents=True, exist_ok=True)
bordeaux_extent = [-1.25, -0.05, 44.35, 45.35]
nearby_cities = {
    "Bordeaux": (-0.5792, 44.8378),
    "Libourne": (-0.2410, 44.9150),
    "Arcachon": (-1.1740, 44.6580),
    "Langon": (-0.2500, 44.5530),
}

# SWI002 is the Soil Water Index with a time constant of T=2.
with tempfile.TemporaryDirectory() as temp_dir:
    with zipfile.ZipFile(zip_path) as archive:
        swi_member = next(
            member
            for member in archive.namelist()
            if "SWI002" in member and member.lower().endswith((".tif", ".tiff"))
        )
        archive.extract(swi_member, temp_dir)

    swi_path = Path(temp_dir) / swi_member
    with rasterio.open(swi_path) as src:
        window = from_bounds(
            bordeaux_extent[0],
            bordeaux_extent[2],
            bordeaux_extent[1],
            bordeaux_extent[3],
            transform=src.transform,
        )
        window = window.intersection(Window(0, 0, src.width, src.height))
        swi = src.read(1, window=window, masked=True).astype("float32")
        swi_transform = src.window_transform(window)
        nodata = src.nodata

    swi_values = swi.filled(np.nan)
    if nodata is not None:
        swi_values[swi_values == nodata] = np.nan

finite_values = swi_values[np.isfinite(swi_values)]
if finite_values.size == 0:
    raise ValueError("No valid SWI pixels were found in the Bordeaux area")

swi_left, swi_bottom, swi_right, swi_top = array_bounds(
    swi_values.shape[0], swi_values.shape[1], swi_transform
)

# -----------------------------
# Figure (same layout as the GEDI, S3, S5P and ERA5 maps)
# -----------------------------
map_projection = ccrs.PlateCarree()
figure, map_ax = plt.subplots(
    figsize=(12, 10),
    subplot_kw={"projection": map_projection},
    constrained_layout=True,
)
map_ax.add_feature(cfeature.OCEAN, facecolor="#dbeaf2", zorder=0)
map_ax.add_feature(cfeature.LAND, facecolor="#eeeeea", zorder=0)
map_ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="#444444", zorder=3)
map_ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="#666666", zorder=3)
map_ax.set_extent(bordeaux_extent, crs=map_projection)

raster = map_ax.imshow(
    swi_values,
    extent=(swi_left, swi_right, swi_bottom, swi_top),
    origin="upper",
    cmap="YlGnBu",
    vmin=0,
    vmax=100,
    transform=map_projection,
    zorder=2,
)

city_longitudes = [location[0] for location in nearby_cities.values()]
city_latitudes = [location[1] for location in nearby_cities.values()]
map_ax.scatter(
    city_longitudes,
    city_latitudes,
    c="red",
    s=50,
    edgecolors="white",
    linewidths=1.2,
    transform=map_projection,
    zorder=4,
)
for city_name, (longitude, latitude) in nearby_cities.items():
    map_ax.text(
        longitude + 0.02,
        latitude + 0.02,
        city_name,
        fontsize=18 if city_name == "Bordeaux" else 14,
        color="black",
        ha="left",
        va="bottom",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 2},
        transform=map_projection,
        zorder=5,
    )

map_ax.set_title("23 July 2026 12:00 UTC", fontsize=16)
map_ax.set_xlabel("Longitude", fontsize=16)
map_ax.set_ylabel("Latitude", fontsize=16)
gridlines = map_ax.gridlines(
    draw_labels=True,
    linewidth=0.4,
    alpha=0.5,
    linestyle="--",
)
gridlines.top_labels = False
gridlines.right_labels = False
gridlines.xlabel_style = {"size": 16}
gridlines.ylabel_style = {"size": 16}

# Colorbar axes in map-axes coordinates, so it matches the map height exactly
colorbar_ax = map_ax.inset_axes([1.03, 0, 0.04, 1])
colorbar = figure.colorbar(raster, cax=colorbar_ax, orientation="vertical")
colorbar.set_label("Bodemvocht Index (%)", fontsize=22)
colorbar.ax.tick_params(labelsize=22)

output_svg = figure_folder / "SWI_20260723.svg"
figure.savefig(output_svg, format="svg", bbox_inches="tight")
plt.show()
plt.close(figure)

print(f"Saved {output_svg}")
print(f"Valid Bordeaux-area pixels: {finite_values.size:,}")
print(f"SWI range in map: {finite_values.min():.1f}% to {finite_values.max():.1f}%")

In [ ]:
from IPython.display import display

# Child-friendly version of the figure above: no numbers on the colorbar,
# just "Laag" at the bottom, "Hoog" at the top and an arrow from low to high.
low_label, high_label = "Laag", "Hoog"

# Remove earlier additions so re-running this cell does not stack them
for artist in [text for text in colorbar.ax.texts if text.get_gid() == "kids_colorbar"]:
    artist.remove()

colorbar.set_ticks([])
colorbar.ax.text(0.5, -0.01, low_label, transform=colorbar.ax.transAxes,
                 ha="center", va="top", fontsize=22, gid="kids_colorbar")
colorbar.ax.text(0.5, 1.01, high_label, transform=colorbar.ax.transAxes,
                 ha="center", va="bottom", fontsize=22, gid="kids_colorbar")
colorbar.ax.annotate(
    "",
    xy=(2.0, 0.97),
    xytext=(2.0, 0.03),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "-|>", "linewidth": 3, "color": "black", "mutation_scale": 30},
    annotation_clip=False,
    gid="kids_colorbar",
)
colorbar.ax.yaxis.labelpad = 45  # keep the colorbar title clear of the arrow

kids_svg = output_svg.with_name(f"{output_svg.stem}_kids.svg")
figure.savefig(kids_svg, format="svg", bbox_inches="tight")
display(figure)
print(f"Saved {kids_svg}")